> 업로드 정리 주석(2026-10-30)
> - 이 노트북은 제공된 코드·해석의 기록이며 이번 정리에서 실제 데이터를 넣어 재실행하지 않았습니다.
> - 현재 코드의 분석 상한은 예측기준월 202602입니다. 아래 해석에 남은 2026.03·2026.02 원자료 등의 옛 수치는 현재 코드 재실행 결과로 확정하면 안 됩니다.
> - 전체 기간 Spearman 상관은 탐색 결과입니다. 최신 학습 구간 Pearson 분석 및 최종 변수 선택과 구분하세요.
> - 카드 13절의 라벨 분석은 202407~202508을 사용합니다. 최신 학습 범위(상관분석 기준 202301~202502)와 달라 모델 선택 근거로 재사용하기 전에 수정·재검증해야 합니다.
> - 낮은 상관만으로 독립성을 입증하거나 변수를 자동 채택하지 않습니다.
> - 실제 입력 데이터와 실행 출력은 공개 업로드 묶음에서 제외했습니다.


# 02. 통신(생활인구) EDA
대상: `통신데이터_전처리v02.csv`(동×월×시간대, 24행/동·월) + `통신데이터_전처리_v03.csv`(동×월, 월 평균과 지표 4개)

**규칙**
- 원자료(거주·직장·방문 인구)는 **2026.01까지만** 본다. (목표: t = 2026.02 예측)
- v03의 지표 4개는 t = 2026.02까지 본다. 2026.02 행이 시험 문제다.
- v03의 `거주인구_월`·`직장인구_월`·`방문인구_월`·`총생활인구_월`·`방문인구_시간대변동계수_월`은 **그 달 자체 값**이라 모델 입력에 쓰지 않는다(EDA 재료용).
- 폐업 라벨과 연결 짓는 분석은 하지 않는다.
- 2025.01·02 녹산동은 추정값(보간)이다. 그 구간이 튀는지 따로 본다.

**구성** 1 불러오기 → 2 월별 분포 → 3 시간대 패턴 → 4 시간 추이·녹산동 → 5 구·동 비교 → 6 방문인구 비중 → 7 지표 분포·추이 → 8 지표 재계산 검증(v02 → v03) → 9 상관 → 10 극단값 → 11 정리

> ⚠ **시험 기준월을 2026.03 → 2026.02로 변경** (자료는 2026.01까지, 정답 기간 2026.02~2026.07). 위 설명과 코드(`CUT` 등)는 새 기준으로 고쳤다. **아래 '결과 해석'과 마지막 정리 표의 숫자·그림 설명은 이전 기준(자료 2026.02까지, 시험 2026.03)으로 쓴 것이라 한 달씩 어긋난다. 다시 실행해서 그림과 숫자를 새로 확인한 뒤 갱신한다.**


In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
plt.rcParams.update({"font.family": "Malgun Gothic", "axes.unicode_minus": False,
                     "figure.dpi": 100, "axes.grid": True, "grid.alpha": .3})
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}".rstrip("0").rstrip("."))   # 8.3e+04 같은 지수 표기 대신 쉼표 숫자로 (소수는 최대 4자리)

V2, V3 = "통신데이터_전처리v02.csv", "통신데이터_전처리_v03.csv"   # 이 노트북과 같은 폴더(완전쓰는거_ver2)에서 실행
CUT = 202601
IND = ["직장인구_6개월평균", "방문인구_시간대변동계수_6개월", "방문인구_전년대비증감률", "방문인구비중"]
POP = ["거주인구_월", "직장인구_월", "방문인구_월"]

## 1. 불러오기와 첫 확인

In [ ]:
v2 = pd.read_csv(V2, encoding="utf-8-sig", low_memory=False).drop(columns=["Unnamed: 0"], errors="ignore")
v3 = pd.read_csv(V3, encoding="utf-8-sig")
v2 = v2[v2.기준월 <= CUT].copy()                 # 시간대별 원자료: 2026.01까지
raw = v3[v3.기준월 <= CUT].copy()                # 월별 원자료: 2026.01까지
der = v3[v3.기준월 <= 202602].copy()             # 지표: 2026.02까지
for x in (raw, der): x["월"] = pd.to_datetime(x.기준월.astype(str), format="%Y%m")

print("v02", v2.shape, "| v03 원자료", raw.shape, "| v03 지표", der.shape)
print("기간", raw.기준월.min(), "~", raw.기준월.max(), "| 동", raw.행정동코드.nunique(), "| 구", raw.구군.nunique())
print("v02 동·월당 시간대 수:", sorted(v2.groupby(["행정동코드", "기준월"]).size().unique()), "| 키 중복(v03)", raw.duplicated(["기준월", "행정동코드"]).sum())
print("원자료 빈칸", int(v2[["거주인구_평균", "직장인구_평균", "방문인구_평균"]].isna().sum().sum()), "| 보간 행(v02)", v2.보간방법.value_counts().to_dict())
raw[POP + ["총생활인구_월"]].describe().round(0)

### 결과 해석
- v02 186,960행(205동 × 38개월 × 24시간), v03 원자료 7,790행(205동 × 38개월), 지표용 7,995행(39개월). 동·월마다 24시간이 모두 있고, 빈칸 0, 키 중복 0.
- 보간(추정) 행은 v02에 48행(녹산동 2025.01 `sinho_delta` 24, 2025.02 `interp` 24)이다.
- **이상한 점:** 없음.

## 2. 월별 인구 분포 (거주·직장·방문, 동×월)

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(13, 3.5))
for a, c in zip(ax, POP):
    a.hist(np.log10(raw[c].clip(lower=1)), bins=60); a.set_title(c + " (log10 명)")
plt.tight_layout(); plt.show()
print("동·월 중앙값(명)", raw[POP].median().round(0).to_dict())
print("직장 인구가 거주 인구보다 큰 동·월 비율", round((raw.직장인구_월 > raw.거주인구_월).mean(), 3))
raw[POP].quantile([0, .01, .5, .99, 1]).round(0).T

### 결과 해석
- 동·월 중앙값은 **거주 9,092명 · 직장 1,232명 · 방문 3,139명**.
- 로그로 그린 이유: 동마다 인구가 수백 배 차이(거주 182명 ~ 64,388명)라서.
- 거주는 3.5~4.4(약 3천 ~ 2.5만 명)에 가장 많고 왼쪽에 작은 동 꼬리가 있다. 직장·방문은 종 모양에 가깝다.
- **이상한 점:** 직장 인구가 거주 인구보다 큰 동·월은 1.3%뿐이다(업무지구). 거주 쪽 오른쪽 끝의 높은 막대는 정관읍 같은 대규모 주거 동네다.

## 3. 시간대(0~23시) 패턴

In [ ]:
hr = v2.groupby("시간")[["거주인구_평균", "직장인구_평균", "방문인구_평균"]].mean()
fig, ax = plt.subplots(1, 3, figsize=(13, 3.5))
for a, c in zip(ax, hr.columns):
    hr[c].plot(ax=a, marker="o", ms=3); a.set_title(c + " (전체 동 평균)"); a.set_xticks(range(0, 24, 3))
plt.tight_layout(); plt.show()
print("시간대별 최대·최소 시각", {c: (int(hr[c].idxmax()), int(hr[c].idxmin())) for c in hr.columns})
(hr.max() / hr.min()).round(2).rename("시간대 최대/최소 비율")

### 결과 해석 (전체 동 평균)
- **거주:** 새벽 4시에 최대(약 13,700명), 오후 2시에 최소(약 7,600명). 낮에 줄고 밤에 늘어난다(최대/최소 1.8배).
- **직장:** 오전 8~9시에 급증해 오후 3시 최대(약 3,500명), 밤 8시 이후 급감해 새벽에는 거의 0이다(최대/최소 19.9배). 시간대 폭이 가장 크다.
- **방문:** 새벽 4시 최소에서 오후 2시 최대(약 5,300명)까지 완만히 오른다(2.0배).
- **이상한 점:** 없음. 사람들이 사는 곳(밤) · 일하는 곳(낮) · 놀러 가는 곳(오후)이라는 상식과 맞다.

In [ ]:
# 동네 성격에 따라 하루 패턴이 어떻게 다른가: 방문 비중 상위·하위 동의 방문 인구 시간대 패턴 (자기 평균을 1로 맞춤)
last = raw[raw.기준월 == CUT]
share = (last.방문인구_월 / last.총생활인구_월)
top = last.loc[share.nlargest(5).index, "행정동코드"]; bot = last.loc[share.nsmallest(5).index, "행정동코드"]
f = v2[v2.기준월 == CUT]
fig, ax = plt.subplots(figsize=(8, 3.5))
for name, codes, ls in [("방문 비중 상위 5동", top, "-"), ("방문 비중 하위 5동", bot, "--")]:
    p = f[f.행정동코드.isin(codes)].groupby("시간").방문인구_평균.mean()
    (p / p.mean()).plot(ax=ax, label=name, ls=ls)
ax.set_title("방문인구 시간대 패턴 (자기 평균 = 1, 2026.01)"); ax.legend(); plt.show()
last.loc[share.nlargest(5).index, ["구군", "읍면동"]].assign(방문비중=share.nlargest(5).round(3))

### 결과 해석
- **상권형 동네는 저녁까지 방문이 이어진다.** 방문비중 상위 5개 동은 오후 3시에 정점을 찍고 밤 9시에도 평균의 1.1배인 반면, 하위 5개 동(주거지)은 낮 1~2시 정점 후 밤 9시에 0.98배로 빨리 내려간다.
- `방문인구_시간대변동계수` 지표가 이런 차이를 잡는다.
- 방문비중이 가장 높은 5개 동: 강서구 대저2동, 중구 남포동, 부산진구 부전2동, 중구 부평동, 중구 광복동.
- **이상한 점:** 없음.

## 4. 시간 추이와 녹산동 (2025.01·02 추정값)

In [ ]:
tot = raw.groupby("월")[POP].sum() / 1e4
tot.plot(figsize=(11, 3.8), title="월별 부산 전체 인구 합계 (만 명, 동별 24시간 평균의 합)"); plt.show()
print("2023.01 → 2026.01 증가율"); (tot.iloc[-1] / tot.iloc[0] - 1).round(3)

### 결과 해석 (월별 부산 전체 인구 합계)
- 거주 약 212~232만 명(2024년 이후 완만한 감소), 직장 약 30~40만 명(감소), 방문 약 75~85만 명(비슷하거나 소폭 증가).
- 2023.01 → 2026.02: **거주 −2.2%, 직장 −8.2%, 방문 +12.7%**. 방문 증가의 일부는 2023.01 값이 가장 낮은 데서 온다.
- **이상한 점:** **2025.01에 수준이 한 단계 이동한다.** 직장 합계가 약 35만 → 29만으로 떨어지고 거주도 낮아졌다가 2025.03 이후 낮은 수준이 유지된다. 2025.01은 신호동이 분리되기(2025.02) 직전 달이다. 통신 기준 변경이나 구역 조정이 원인일 가능성이 있어 확인이 필요하다.

In [ ]:
ns = raw[raw.읍면동 == "녹산동"].set_index("월")[POP]
fig, ax = plt.subplots(figsize=(10, 3.5)); ns.plot(ax=ax, marker="o", ms=3)
for d_ in ["2025-01-01", "2025-02-01"]: ax.axvline(pd.Timestamp(d_), color="r", ls="--", alpha=.5)
ax.set_title("녹산동 월별 인구 (빨간선: 2025.01 sinho_delta, 2025.02 interp 추정값)"); plt.show()
ns.loc["2024-11":"2025-05"].round(0)

### 결과 해석 (녹산동, 빨간 선 = 2025.01 · 02 추정값)
- **2025.03에 거주인구가 계단처럼 떨어진다.** 25,590명(2025.01) → 22,681명(02, 추정) → 19,772명(03)으로 **−23%**이고, 이후 약 20,000명에서 유지된다.
- 직장인구는 2025.03 이후 1.2~1.5만 명 사이에서 달마다 크게 출렁인다(합산 전에는 완만). 방문인구는 합산 후 소폭 올라간다(약 11,600 → 13,000).
- **이상한 점:** 신호동을 합산했는데도 거주인구가 크게 줄었다. 신호동 분리 때 일부 구역이 다른 동으로 옮겨졌을 가능성이 있다. 합산 후 값이 이전 값과 같은 기준인지는 확인이 필요하다.

## 5. 구·동 비교 (2026.02)

In [ ]:
g = last.groupby("구군")[POP].sum() / 1e4
g.sort_values("거주인구_월").plot.barh(figsize=(10, 5), title="구별 인구 합계 (만 명, 2026.01)"); plt.show()
print("상위 5 동 (총생활인구)"); last.nlargest(5, "총생활인구_월")[["구군", "읍면동"] + POP].round(0)

### 결과 해석 (2026.02)
- 구별 거주인구는 **해운대구(약 25.6만 명)**, 부산진구(23만), 사하구(21만), 북구(18.5만) 순으로 크다. 중구(약 2.6만)가 가장 작다.
- 방문인구는 해운대구·부산진구가 약 10만 명으로 가장 크다. 직장인구는 해운대구·부산진구(약 3.2만)와 강서구(약 2.9만)가 크다.
- 총생활인구 상위 동: 기장군 정관읍(거주 60,590명), 기장읍, 강서구 명지1동, 강서구 녹산동, 해운대구 우2동.
- **이상한 점:** 강서구는 직장 인구가 거주에 비해 많은 편이다(녹산산업단지 영향으로 보임, 산업단지 여부는 확인하지 못했다).

## 6. 방문인구 비중 (방문 ÷ 거주+직장+방문)

In [ ]:
der["방문비중_월"] = der.방문인구_월 / der.총생활인구_월
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].hist(der.방문비중_월.dropna(), bins=60); ax[0].set_title("방문비중 (동×월, 그 달 자체 값)")
der.groupby("월").방문비중_월.median().plot(ax=ax[1], title="월별 중앙값"); plt.tight_layout(); plt.show()
l2 = der[der.기준월 == CUT].assign(비중=lambda x: x.방문비중_월).sort_values("비중")
print("방문비중 하위 5"); print(l2.head(5)[["구군", "읍면동", "비중"]].round(3).to_string(index=False))
print("방문비중 상위 5"); print(l2.tail(5)[["구군", "읍면동", "비중"]].round(3).to_string(index=False))
print("0~1 범위 밖:", ((der.방문비중_월 < 0) | (der.방문비중_월 > 1)).sum())

### 결과 해석
- 방문비중(방문 ÷ 총생활인구)은 0.1~0.3에 대부분 몰려 있고 오른쪽 꼬리가 길다. 0~1 범위를 벗어나는 값은 0건.
- 높은 곳: 강서구 대저2동 0.70, 중구 남포동 0.68, 부산진구 부전2동 0.59. 낮은 곳: 해운대구 반송2동·기장군 정관읍 0.10, 사하구 다대2동 0.11.
- **이상한 점:** 월별 중앙값이 **2025.01부터 0.215 → 0.24 수준으로 올라가** 이전보다 높은 쪽에서 오르내린다(위 4번의 2025.01 수준 이동과 같은 시기).

## 7. 지표 4개 분포와 추이 (t = 2023.01 ~ 2026.03)

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(11, 6)); ax = ax.ravel()
for a, c in zip(ax, IND):
    v = der[c].dropna(); a.hist(v.clip(v.quantile(.01), v.quantile(.99)), bins=60); a.set_title(c + " (1~99% 구간)")
plt.tight_layout(); plt.show()
fig, ax = plt.subplots(2, 2, figsize=(11, 6)); ax = ax.ravel()
for a, c in zip(ax, IND): der.groupby("월")[c].median().plot(ax=a); a.set_title(c + " (동 중앙값)")
plt.tight_layout(); plt.show()
first = {c: int(der.loc[der[c].notna(), "기준월"].min()) for c in IND}
print("첫 값이 있는 달", first, "(기대: 2023.07 / 2023.07 / 2024.07 / 2023.07)")
der[IND].describe(percentiles=[.01, .05, .5, .95, .99]).T.round(3)

### 결과 해석 (위: 분포, 아래: 동 중앙값의 시간 추이)
- **직장인구_6개월평균:** 0~2,000에 몰려 있고 오른쪽 꼬리가 길다(최대 16,853). 중앙값은 1,366 → 1,147로 **계속 감소**.
- **방문인구_시간대변동계수:** 0.1~0.35에 넓게 퍼져 있다. 중앙값은 0.216에서 0.245까지 오르다가 **2025.09경 0.225로 뚝 떨어진다.**
- **방문인구_전년대비증감률:** 0을 중심으로 종 모양. 2024.12에 저점(−0.07)이었다가 2026.01에 +0.06으로 반등한다.
- **방문인구비중:** 0.1~0.3에 몰려 있고 오른쪽 꼬리가 길다. 중앙값은 2025년부터 0.218 → 0.235로 상승한다.
- 지표별 첫 값이 있는 달은 명세서의 시작 t와 같다(2023.07 / 2023.07 / 2024.07 / 2023.07).
- **이상한 점:** 방문 변동계수가 2025.09경에 갑자기 떨어지는 이유는 확인하지 못했다(명절·행사 영향 가능).

## 8. 지표 재계산 검증 (v02에서 다시 계산해서 v03과 비교)
시간대별 원자료(v02)로 4개 지표를 직접 다시 계산한 값이 v03의 열과 같은지 확인한다.

In [ ]:
months = sorted(v2.기준월.unique()); dongs = sorted(v2.행정동코드.unique()); M = len(months); D = len(dongs)
def cube(col):                                   # (동, 월, 24시간) 배열
    ix = pd.MultiIndex.from_product([dongs, months, range(24)], names=["행정동코드", "기준월", "시간"])
    return v2.set_index(["행정동코드", "기준월", "시간"])[col].reindex(ix).to_numpy(float).reshape(D, M, 24)
Hc, Wc, Vc = cube("거주인구_평균"), cube("직장인구_평균"), cube("방문인구_평균")
assert not np.isnan(Vc).any()
Hm, Wm, Vm = Hc.mean(2), Wc.mean(2), Vc.mean(2)                 # 동×월 24시간 평균

ts = pd.period_range("2023-01", "2026-02", freq="M").strftime("%Y%m").astype(int); T = len(ts)
w6, cv6, yoy, vs = (np.full((D, T), np.nan) for _ in range(4))
for i in range(6, T):                                            # t = ts[i], 사용 월 = i-6 ~ i-1 (월 번호는 2023.01 = 0)
    w6[:, i] = Wm[:, i-6:i].mean(1)
    vh = Vc[:, i-6:i, :].mean(1); cv6[:, i] = vh.std(1) / vh.mean(1)          # 24개 시간대 6개월 평균의 변동계수 (ddof=0)
    vs[:, i] = Vm[:, i-6:i].mean(1) / (Hm[:, i-6:i].mean(1) + Wm[:, i-6:i].mean(1) + Vm[:, i-6:i].mean(1))
    if i >= 18: yoy[:, i] = Vm[:, i-6:i].sum(1) / Vm[:, i-18:i-12].sum(1) - 1

mine = {"직장인구_6개월평균": w6, "방문인구_시간대변동계수_6개월": cv6, "방문인구_전년대비증감률": yoy, "방문인구비중": vs}
theirs = {c: der.pivot(index="행정동코드", columns="기준월", values=c).reindex(index=dongs, columns=ts).to_numpy(float) for c in IND}
rows = []
for c in IND:
    a, b = mine[c], theirs[c]
    both = ~np.isnan(a) & ~np.isnan(b)
    rows.append({"지표": c, "비교 칸 수": int(both.sum()), "최대 절대차": float(np.abs(a - b)[both].max()),
                 "한쪽만 빈칸": int((np.isnan(a) != np.isnan(b)).sum())})
chk = pd.DataFrame(rows); print(chk.to_string(index=False))
# v03은 직장인구 평균을 소수 1자리, 나머지 지표를 소수 4자리로 반올림해 저장했다 → 허용 오차 = 반올림 단위의 절반
tol = chk["지표"].map(lambda c: 0.05 if c == "직장인구_6개월평균" else 0.00005) + 1e-9
assert (chk["최대 절대차"].values <= tol.values).all() and (chk["한쪽만 빈칸"] == 0).all()
print("→ 4개 지표 모두 v03과 일치 (차이는 v03의 반올림 범위 이내)")

### 결과 해석
- v02(시간대별)에서 4개 지표를 직접 다시 계산한 값이 **v03과 일치한다.** 최대 차이는 직장인구 0.05명, 나머지 0.00005로, v03이 소수 1자리·4자리로 저장한 반올림 범위 안이다. 한쪽만 비어 있는 칸도 0건이다.
- 비교 칸 수는 직장·변동계수·방문비중 6,765개, 방문 전년비 4,305개다.
- **이상한 점:** 없음. v03 지표는 믿고 쓸 수 있다.

## 9. 지표끼리 상관 (Spearman)

In [ ]:
cr = der[IND].corr(method="spearman")
fig, ax = plt.subplots(figsize=(5.5, 4.5))
im = ax.imshow(cr, vmin=-1, vmax=1, cmap="RdBu_r"); ax.grid(False)
ax.set_xticks(range(4), IND, rotation=45, ha="right"); ax.set_yticks(range(4), IND)
for i in range(4):
    for j in range(4): ax.text(j, i, f"{cr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=9)
plt.colorbar(im); plt.tight_layout(); plt.show()
print("상관 0.8 초과 쌍:", [(a, b, round(cr.loc[a, b], 2)) for i, a in enumerate(IND) for b in IND[i+1:] if abs(cr.loc[a, b]) > .8] or "없음")
print(cr.round(2).to_string())

### 결과 해석
- **상관이 0.8을 넘는 쌍이 없다.** 가장 높은 쌍은 직장인구_6개월평균 × 방문인구_시간대변동계수 0.38이다.
- 나머지 쌍은 대부분 0.1 안팎으로 서로 독립적이다.
- **이상한 점:** 없음.

## 10. 극단값과 녹산동 추정 구간

In [ ]:
t = der[der.기준월 == 202602]
for c in IND:
    print(f"== {c} 상위 3 / 하위 3 (2026.02)")
    print(pd.concat([t.nlargest(3, c), t.nsmallest(3, c)])[["구군", "읍면동", c]].round(3).to_string(index=False))
print("\n녹산동 지표 (추정값이 t−1 구간에 들어가는 달)")
der[der.읍면동 == "녹산동"].set_index("기준월").loc[202412:202508, IND].round(3)

### 결과 해석 (2026.03)
- **직장인구_6개월평균:** 상위는 강서구 녹산동(13,535), 해운대구 우2동(8,204), 기장군 정관읍(7,005). 하위는 동구 초량6동(85), 수정4동(91), 중구 영주1동(99).
- **방문 변동계수:** 상위는 중구 광복동 0.82, 금정구 금성동 0.79, 선두구동 0.72(특정 시간대에만 몰림). 하위는 연제구 연산8동 0.032, 동래구 온천3동 0.035, 남구 문현3동 0.038(하루 종일 비슷함).
- **방문 전년비:** 상위는 **강서구 강동동 +1.158**, 서구 동대신2동 +0.47, 북구 덕천3동 +0.42. 하위는 동구 초량6동 −0.548.
- **방문비중:** 상위는 중구 남포동 0.68, 강서구 대저2동 0.67. 하위는 해운대구 반송2동 0.088, 기장군 정관읍 0.093.
- **녹산동 추정값 구간:** 2025.01·02가 t−1 구간에 들어가는 달의 지표(직장 약 15,200~15,400, 변동계수 약 0.48, 방문비중 약 0.22)가 앞뒤 달과 이어진다. **추정값 때문에 지표가 튀지는 않는다.**
- **이상한 점:** 강동동의 방문 전년비 +116%는 카드에서도 극단값이었던 동네와 같다(03번 노트북 7번).

## 11. 이상한 점과 이유 (정리)

각 셀 아래 '결과 해석'에 그림을 보고 읽은 내용을 적었다. 이 표는 핵심 수치만 모은 것이다.

| 항목 | 확인한 점 | 이유·해석 |
|---|---|---|
| 기본 점검 | v02 186,960행(205동 × 38개월 × 24시간), 동·월마다 24시간 완비, 빈칸 0, 키 중복 0 | 이상 없음 |
| 인구 규모 | 동·월 중앙값 거주 9,092명 / 직장 1,232명 / 방문 3,139명 | 직장 > 거주인 동·월은 1.3%뿐 (업무지구) |
| 시간대 패턴 | 거주 새벽 4시 최대·오후 2시 최소(1.8배), 직장 오후 3시 최대·새벽 3시 최소(19.9배), 방문 오후 2시 최대·새벽 4시 최소(2.0배) | 직장 인구의 시간대 폭이 가장 크다 |
| 3년 변화 | 2023.01 → 2026.02 거주 −2.2%, 직장 −8.2%, 방문 +12.7% | 방문은 늘고 직장·거주는 줄었다 |
| 방문 비중 | 높은 곳: 강서구 대저2동 0.70, 중구 남포동 0.68, 부산진구 부전2동 0.59. 낮은 곳: 해운대구 반송2동·기장군 정관읍 0.10 | 상권·관광지는 높고 주거지는 낮다. 0~1 범위를 벗어나는 값 없음 |
| 지표 시작 달 | 직장·변동계수·방문비중 2023.07, 방문 전년비 2024.07 | 명세서와 일치 |
| 지표 재계산 | v02로 다시 계산한 4개 지표가 v03과 일치 (차이는 반올림 범위 이내: 직장 0.05명, 나머지 0.00005) | v03 지표가 맞게 계산돼 있다 |
| 상관 | 0.8 초과 쌍 없음. 가장 높은 쌍은 직장인구 × 변동계수 0.38 | 하나만 남겨야 할 지표 없음 |
| 방문 전년비 극단 | 강서구 강동동 +1.158, 동구 초량6동 −0.548 | 강동동은 카드 EDA에서도 전년비 극단값(자동차 +600% 등)이었다. 두 데이터에서 같이 튀므로 실제 변화(개발 등) 가능성이 있어 확인 필요 |
| 변동계수 극단 | 높음: 중구 광복동 0.82, 금정구 금성동 0.79, 선두구동 0.72 | 특정 시간대에만 방문이 몰리는 동네 |
| 녹산동 합산 | 거주인구 2025.01 25,590 → 2025.03 19,772명 (−23%), 직장 −11%, 방문 +12% | 신호동 합산 후에도 거주가 크게 낮아졌다. 분리 때 구역이 다른 동으로 옮겨졌을 가능성이 있어 확인 필요 |
| 녹산동 지표 | 2025.01·02(추정값) 구간의 지표(직장 평균 약 15,300, 변동계수 0.48, 방문비중 0.22)가 앞뒤 달과 이어진다 | 추정값 때문에 튀지는 않는다 |